# Baseline Experiments for the exchange rate dataset
#### Objective:The baseline model for the prediction project is established here
#### Metric:  sMAPE
#### Forecast-horizon: 24-step and 1-step

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [4]:
df_train = pd.read_csv("../data/raw/train_data.csv")
df_val = pd.read_csv("../data/raw/validation_data.csv")

FEATURE_COLS = [c for c in df_train.columns if c != "timestamp_idx"]

train_data = df_train[FEATURE_COLS].values
val_data = df_val[FEATURE_COLS].values

print("Train shape:", train_data.shape)


Train shape: (5311, 9)


In [5]:
def create_windows(data, seq_len, pred_len):
    X, Y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i:i+seq_len])
        Y.append(data[i+seq_len:i+seq_len+pred_len])
    return np.array(X), np.array(Y)


 Before building any model, compute naive persistence baseline

In [6]:
def smape(y_true, y_pred):
    eps = 1e-8
    return torch.mean(
        2 * torch.abs(y_true - y_pred) /
        (torch.abs(y_true) + torch.abs(y_pred) + eps)
    )


## Experiment 1

#### Here a full Transformer model is used to do raw level prediction 